# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # No heavy dependencies needed for this conceptual demo
    pass
    print('Setup complete.')


# Agentic AI Concepts: The ReAct Pattern In-Depth

Agentic AI goes beyond simple classification or prediction. Agents can *reason* about a problem, *act* by using tools, and *observe* the results to adjust their plan. This pattern is known as **ReAct (Reasoning + Acting)**.

In this notebook, we build a **conceptual Agent** from scratch. Instead of using a paid API (like OpenAI), we use a "Mock LLM"—a simple function that simulates how a language model would respond to prompts—to demonstrate the architecture.

## 1. Defining Tools
Agents need tools to interact with the world. Here are two simple Python functions.

In [ ]:
def search_database(query):
    """Simulates searching a database."""
    print(f"[Tool] Searching database for: {query}")
    if "Alice" in query:
        return "User: Alice, ID: 8821"
    elif "Bob" in query:
        return "User: Bob, ID: 9912"
    return "No results found."

def get_transactions(user_id):
    """Simulates retrieving financial records."""
    print(f"[Tool] Getting transactions for ID: {user_id}")
    if user_id == "8821":
        return "Last purchase: $120.50 at Amazon."
    elif user_id == "9912":
        return "Last purchase: $5.00 at CoffeeShop."
    return "No transactions."

# Registry of available tools
tools = {
    "search_database": search_database,
    "get_transactions": get_transactions
}

## 2. The "Mock" LLM
In a real application, this would be a call to GPT-4 or Llama 2. Here, we simulate the *reasoning process* with simple if-then rules based on the prompt history. The key is to see the format: `Thought` -> `Action`.

In [ ]:
def mock_llm_completion(prompt):
    """
    Simulates an LLM's response based on the conversation history.
    Returns the next step (Thought + Action or Final Answer).
    """
    last_line = prompt.strip().split('\n')[-1]
    
    # Scenario: User asks about Alice
    if "Question: How much did Alice spend" in prompt and "User: Alice" not in prompt:
        return "Thought: I need to find Alice's user ID first.\nAction: search_database[Alice]"
    
    elif "User: Alice, ID: 8821" in prompt and "Last purchase" not in prompt:
        return "Thought: I have the ID (8821). Now I can look up her transactions.\nAction: get_transactions[8821]"
    
    elif "Last purchase: $120.50" in prompt:
        return "Thought: I have the transaction info. I can answer the user.\nFinal Answer: Alice spent $120.50 on her last purchase."
    
    return "Error: Confused."

## 3. The Agent Loop (ReAct)
This is the core logic. The agent:
1.  Reads the history.
2.  Decides what to do (Thought + Action).
3.  Parses the action.
4.  Executes the tool.
5.  Adds the observation to history.
6.  Repeats.

In [ ]:
import re

class ReActAgent:
    def __init__(self, tools, llm_func):
        self.tools = tools
        self.llm = llm_func
        self.history = ""
        
    def run(self, question):
        self.history = f"Question: {question}\n"
        print(f"Goal: {question}")
        print("-" * 30)
        
        for i in range(5): # Limit steps to prevent infinite loops
            # 1. Ask LLM for next step
            response = self.llm(self.history)
            print(f"\033[94m{response}\033[0m") # Print in blue
            self.history += response + "\n"
            
            # 2. Check for Final Answer
            if "Final Answer:" in response:
                return response.split("Final Answer:")[1].strip()
            
            # 3. Parse Action using Regex
            # Looking for Action: tool_name[argument]
            match = re.search(r"Action: (\w+)\[(.*?)\]", response)
            if match:
                tool_name = match.group(1)
                tool_arg = match.group(2)
                
                # 4. Execute Tool
                if tool_name in self.tools:
                    observation = self.tools[tool_name](tool_arg)
                    obs_str = f"Observation: {observation}"
                    print(f"\033[92m{obs_str}\033[0m") # Print in green
                    self.history += obs_str + "\n"
                else:
                    self.history += f"Observation: Tool {tool_name} not found.\n"
            else:
                self.history += "Observation: Could not parse action.\n"
                
        return "Timed out."

# Instantiate and Run
agent = ReActAgent(tools, mock_llm_completion)
result = agent.run("How much did Alice spend on her last purchase?")
print("-" * 30)
print(f"Result: {result}")

## Conclusion
Notice how the agent broke down the problem:
1.  Realized it didn't know Alice's ID.
2.  Called `search_database` to find it.
3.  Used the ID to call `get_transactions`.
4.  Synthesized the final answer.

This loop allows LLMs to solve multi-step problems autonomously.